## Library

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import glob
import numpy as np
from joblib import Parallel, delayed
import time
from shapely.geometry import LineString, Polygon
from shapely.ops import unary_union
from exactextract import exact_extract
import rasterio
import os
from pathlib import Path
from IPython.display import clear_output

## Buffer river width

### Load SWORD reaches

In [ ]:
def load_sword_reaches(sword_path):
    """Load and merge all SWORD reaches shapefiles from NA directory."""
    reaches_pattern = str(Path(sword_path) / 'NA' / '*reaches*.shp')
    reaches_files = sorted(glob.glob(reaches_pattern))
    
    if not reaches_files:
        raise FileNotFoundError(f"No reaches shapefiles found in {sword_path}")
    
    # Read and merge all reaches
    reaches_list = [gpd.read_file(f) for f in reaches_files]
    reaches = gpd.GeoDataFrame(
        pd.concat(reaches_list, ignore_index=True),
        crs=reaches_list[0].crs
    )
    
    return reaches

In [ ]:
# Load reaches
sword_path = '/nas/cee-ice/data/SWORD/SWORD_v17b/shp'
reaches = load_sword_reaches(sword_path)

print(f"Total reaches: {len(reaches)}")
print(f"Columns: {reaches.columns.tolist()}")
print(f"CRS: {reaches.crs}")
print(f"\nWidth statistics:")
print(reaches['width'].describe())

In [ ]:
# Remove reaches with invalid width
reaches_v1 = reaches[~reaches['width'].isin([0, -1])].copy()

print(f"Remaining reaches: {len(reaches_v1)} / {len(reaches)}")

output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp"
reaches_v1.to_file(output_path, driver="Shapefile")

In [ ]:
print(f"\nWidth statistics:")
print(reaches_v1['width'].describe())

### Remove reaches where width > max_width (anomalous) [reaches_v1 -> reaches_v2]

In [62]:
# Remove reaches where width > max_width (anomalous)
invalid_width = reaches_v1['width'] > reaches_v1['max_width']
n_invalid = invalid_width.sum()
pct_invalid = (n_invalid / len(reaches_v1)) * 100

reaches_v2 = reaches_v1[~invalid_width].copy()

print(f"Removed reaches with width > max_width: {n_invalid} ({pct_invalid:.2f}%)")
print(f"Remaining reaches: {len(reaches_v2)} / {len(reaches)} ({len(reaches_v2)/len(reaches)*100:.2f}%)")

Removed reaches with width > max_width: 1327 (3.45%)
Remaining reaches: 37151 / 38696 (96.01%)


In [64]:
print(f"\nWidth statistics:")
print(reaches_v2['width'].describe())


Width statistics:
count    37151.000000
mean       604.950519
std       2191.102027
min          8.611209
25%         63.000000
50%        127.000000
75%        402.000000
max      69822.890625
Name: width, dtype: float64


### (Optional) Dealing with missing width and zero width

In [ ]:
# Find rows with missing width
null_width_rows = reaches[reaches['width'].isnull()]
print(f"Rows with null width: {len(null_width_rows)}")
print(null_width_rows[['reach_id', 'reach_len', 'width', 'wse', 'slope']])

In [18]:
# How many reaches have width = -1?
null_width = reaches[reaches['width'] == -1]
print(f"Reaches with width = -1: {len(null_width)}")
print(f"Percentage: {len(null_width) / len(reaches) * 100:.2f}%")

# Check their characteristics
print("\nCharacteristics of width = -1 reaches:")
print(null_width[['reach_id', 'type', 'lakeflag', 'wse', 'slope']].head(2))

Reaches with width = -1: 209
Percentage: 0.54%

Characteristics of width = -1 reaches:
         reach_id  type  lakeflag   wse         slope
8832  72555100163     3         1  72.0  4.000000e-15
8840  72555100073     3         1  72.0  7.000000e-15


In [19]:
# How many reaches have width = -1?
null_width = reaches[reaches['width'] == 0]
print(f"Reaches with width = 0: {len(null_width)}")
print(f"Percentage: {len(null_width) / len(reaches) * 100:.2f}%")

# Check their characteristics
print("\nCharacteristics of width = 0 reaches:")
print(null_width[['reach_id', 'type', 'lakeflag', 'wse', 'slope']].head(2))

Reaches with width = 0: 9
Percentage: 0.02%

Characteristics of width = 0 reaches:
          reach_id  type  lakeflag   wse     slope
10672  72409000735     5         0  29.0  0.917232
10673  72409000745     5         0  25.0  3.071540


In [33]:
def fill_invalid_widths_iterative(reaches, max_iterations=10):
    """Iteratively fill invalid widths (-1 or 0 m) using average of upstream and downstream."""
    
    reaches_v = reaches.copy()
    
    for iteration in range(max_iterations):
        invalid = reaches_v[reaches_v['width'].isin([0, -1])]
        filled_count = 0
        
        for idx, row in invalid.iterrows():
            up_ids = [int(x) for x in str(row['rch_id_up']).split() if x.strip() and x != 'None'] if pd.notna(row['rch_id_up']) else []
            dn_ids = [int(x) for x in str(row['rch_id_dn']).split() if x.strip() and x != 'None'] if pd.notna(row['rch_id_dn']) else []
            
            upstream = reaches_v[reaches_v['reach_id'].isin(up_ids)]
            valid_up = upstream[~upstream['width'].isin([0, -1])]
            
            downstream = reaches_v[reaches_v['reach_id'].isin(dn_ids)]
            valid_dn = downstream[~downstream['width'].isin([0, -1])]
            
            # Fill only if both upstream and downstream have valid width
            if len(valid_up) > 0 and len(valid_dn) > 0:
                closest_up = valid_up.loc[(valid_up['reach_id'] - row['reach_id']).abs().idxmin()]
                closest_dn = valid_dn.loc[(valid_dn['reach_id'] - row['reach_id']).abs().idxmin()]
                
                avg_width = (closest_up['width'] + closest_dn['width']) / 2
                reaches_v.loc[idx, 'width'] = avg_width
                filled_count += 1
        
        print(f"Iteration {iteration + 1}: Filled {filled_count} reaches")
        if filled_count == 0:
            break
    
    return reaches_v

reaches_v1 = fill_invalid_widths_iterative(reaches)
print(f"\nRemaining invalid widths: {(reaches_v1['width'].isin([0, -1])).sum()}")

Iteration 1: Filled 29 reaches
Iteration 2: Filled 0 reaches

Remaining invalid widths: 189


In [34]:
# Remove reaches with invalid width
reaches_v2 = reaches_v1[~reaches_v1['width'].isin([0, -1])].copy()

print(f"Remaining reaches: {len(reaches_v2)} / {len(reaches_v1)}")

Remaining reaches: 38507 / 38696


### Buffer river width

#### Test manual buffers for acute angle and double-removal cases

In [25]:
# Load and project data
gdf = reaches_v1[reaches_v1["reach_id"] == 71224100423].copy()
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
gdf = gdf.to_crs("EPSG:5070")
crs_meters = gdf.crs

half_width = gdf["width"].iloc[0] * 5.0
linestring = gdf.geometry.iloc[0]
coords = np.array(linestring.coords, dtype=float)  # Convert to numpy array directly

# Helper function to create corner arc at turning points
# Reason: Eliminates duplicate code
def create_corner_arc(p_prev, p_curr, p_next, perp_direction, half_width, num_points=40):
    """
    Create circular arc polygon at corner (turning point).
    
    Args:
        p_prev, p_curr, p_next: coordinate arrays (previous, current, next points)
        perp_direction: perpendicular direction ("left" or "right")
        half_width: buffer distance (half width of corridor)
        num_points: number of arc discretization points (higher = smoother)
    
    Returns:
        Polygon or None if invalid
        
    Reason for function:
    - Reduces code duplication (LEFT and RIGHT arcs use same logic)
    - Improves maintainability (single point of modification)
    - Makes arc generation logic explicit and reusable
    """
    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    
    v1_len = np.linalg.norm(v1)
    v2_len = np.linalg.norm(v2)
    
    # Skip degenerate segments (zero-length or near-zero)
    if v1_len < 1e-10 or v2_len < 1e-10:
        return None
    
    v1_unit = v1 / v1_len
    v2_unit = v2 / v2_len
    
    # Calculate perpendicular vectors
    perp1_base = np.array([-v1_unit[1], v1_unit[0]])
    perp2_base = np.array([-v2_unit[1], v2_unit[0]])
    
    # Apply direction (left = base, right = negative)
    if perp_direction == "right":
        perp1_base = -perp1_base
        perp2_base = -perp2_base
    
    # Create arc by interpolating between two perpendicular directions
    arc_points = []
    for t in np.linspace(0, 1, num_points):
        perp_interp = (1 - t) * perp1_base + t * perp2_base
        perp_norm = np.linalg.norm(perp_interp)
        
        if perp_norm > 1e-10:
            arc_points.append(p_curr + (perp_interp / perp_norm) * half_width)
    
    # Close polygon by returning to center point
    arc_points.append(p_curr)
    
    if len(arc_points) < 3:
        return None
    
    try:
        arc = Polygon(arc_points)
        # Auto-fix self-intersecting polygons
        # Reason: Extreme angles (< 30°) can produce invalid polygon geometry
        if not arc.is_valid:
            arc = arc.buffer(0)
        # Only return geometries with meaningful area
        # Reason: Filter out degenerate polygons from numerical errors
        if arc.is_valid and arc.area > 1e-6:
            return arc
    except:
        pass
    
    return None

# Step 1: Create segment-by-segment buffers
# Reason: Avoids Shapely buffer() limitation with sharp angles
# Each segment is buffered independently with perfect rectangular coverage
segment_buffers = [
    LineString([coords[i], coords[i+1]]).buffer(half_width, cap_style=2, join_style=1)
    for i in range(len(coords) - 1)
]

# Step 2: Create corner arcs (LEFT and RIGHT) at each turning point
# Reason: Segment buffers alone miss coverage at extreme angles;
# corner arcs fill the gap at reflex angles without duplication
corner_arcs = []
for i in range(1, len(coords) - 1):
    for direction in ["left", "right"]:
        arc = create_corner_arc(coords[i-1], coords[i], coords[i+1], direction, half_width)
        if arc is not None:
            corner_arcs.append(arc)

print(f"Segment buffers: {len(segment_buffers)}")
print(f"Corner arcs: {len(corner_arcs)}")

# Step 3: Union in sequential order to preserve geometry integrity
# Reason: Prevents "subtraction effect" by unioning similar geometries first
# Order: segments → arcs (not all-at-once)
# This ensures arc relationships are established before interaction with segments
segment_union = unary_union(segment_buffers)
arc_union = unary_union(corner_arcs) if corner_arcs else None
buffer_geom = unary_union([segment_union, arc_union]) if arc_union else segment_union

# Final validation and auto-fix
# Reason: extreme numerical operations may create invalid geometries
if not buffer_geom.is_valid:
    buffer_geom = buffer_geom.buffer(0)

# Set geometry and metadata
gdf["buffer_geometry"] = buffer_geom
gdf = gdf.set_geometry("buffer_geometry")
gdf.crs = crs_meters

print(f"Final buffer area: {buffer_geom.area:,.0f} m²\n")

# Save to file
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reach_71224100423_complete_buffer.gpkg"
gdf[["reach_id", "buffer_geometry"]].to_file(output_path, driver="GPKG")
print(f"Saved (meters): {output_path}")

output_path_wgs84 = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reach_71224100423_complete_buffer_wgs84.gpkg"
gdf.to_crs("EPSG:4326")[["reach_id", "buffer_geometry"]].to_file(output_path_wgs84, driver="GPKG")
print(f"Saved (WGS84): {output_path_wgs84}")

✓ Saved (meters)
✓ Saved (WGS84)


#### Parallel manual buffers

In [ ]:
# Load and project data
gdf = reaches_v1.copy()
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")

gdf = gdf.to_crs("EPSG:5070")
crs_meters = gdf.crs

print("="*70)
print(f"Processing {len(gdf)} river reaches with parallel buffering")
print("="*70)

# Helper function to create corner arc at turning points
def create_corner_arc(p_prev, p_curr, p_next, perp_direction, half_width, num_points=40):
    """
    Create circular arc polygon at corner (turning point).
    
    Args:
        p_prev, p_curr, p_next: coordinate arrays (previous, current, next points)
        perp_direction: perpendicular direction ("left" or "right")
        half_width: buffer distance (half width of corridor)
        num_points: number of arc discretization points (higher = smoother)
    
    Returns:
        Polygon or None if invalid
    """
    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    
    v1_len = np.linalg.norm(v1)
    v2_len = np.linalg.norm(v2)
    
    if v1_len < 1e-10 or v2_len < 1e-10:
        return None
    
    v1_unit = v1 / v1_len
    v2_unit = v2 / v2_len
    
    perp1_base = np.array([-v1_unit[1], v1_unit[0]])
    perp2_base = np.array([-v2_unit[1], v2_unit[0]])
    
    if perp_direction == "right":
        perp1_base = -perp1_base
        perp2_base = -perp2_base
    
    arc_points = []
    for t in np.linspace(0, 1, num_points):
        perp_interp = (1 - t) * perp1_base + t * perp2_base
        perp_norm = np.linalg.norm(perp_interp)
        
        if perp_norm > 1e-10:
            arc_points.append(p_curr + (perp_interp / perp_norm) * half_width)
    
    arc_points.append(p_curr)
    
    if len(arc_points) < 3:
        return None
    
    try:
        arc = Polygon(arc_points)
        if not arc.is_valid:
            arc = arc.buffer(0)
        if arc.is_valid and arc.area > 1e-6:
            return arc
    except:
        pass
    
    return None

# Main buffering function for each reach
# Reason: Can be parallelized with joblib for 4-8x speedup
def process_reach(reach_id, linestring, width):
    """
    Apply accurate corridor buffer to single reach.
    
    Args:
        reach_id: reach identifier
        linestring: LineString geometry
        width: river width (buffer distance = width * 5)
    
    Returns:
        tuple: (reach_id, buffer_geometry) or (reach_id, None) if processing fails
        
    Reason for wrapper:
    - Enables parallel processing (each reach independent)
    - Isolates error handling (single reach failure doesn't crash entire process)
    - Returns tuple for efficient collection into GeoDataFrame
    """
    try:
        if linestring.is_empty or linestring.length == 0:
            return (reach_id, None)
        
        half_width = width * 5.0
        # half_width = width * 0.5 
        
        coords = np.array(linestring.coords, dtype=float)
        
        # Create segment-by-segment buffers
        segment_buffers = [
            LineString([coords[i], coords[i+1]]).buffer(half_width, cap_style=2, join_style=1)
            for i in range(len(coords) - 1)
        ]
        
        # Create corner arcs at turning points
        corner_arcs = []
        for i in range(1, len(coords) - 1):
            for direction in ["left", "right"]:
                arc = create_corner_arc(coords[i-1], coords[i], coords[i+1], direction, half_width)
                if arc is not None:
                    corner_arcs.append(arc)
        
        # Union segments first, then arcs, then combine
        # Reason: Prevents "subtraction effect" by unioning similar geometries first
        segment_union = unary_union(segment_buffers)
        arc_union = unary_union(corner_arcs) if corner_arcs else None
        buffer_geom = unary_union([segment_union, arc_union]) if arc_union else segment_union
        
        # Final validation
        if not buffer_geom.is_valid:
            buffer_geom = buffer_geom.buffer(0)
        
        return (reach_id, buffer_geom)
    
    except Exception as e:
        print(f"  ERROR: reach {reach_id} - {str(e)}")
        return (reach_id, None)

# Execute parallel processing
# Reason: 40,000 reaches require parallel computation
# n_jobs=-1 uses all CPU cores (~4-8x speedup on typical hardware)
print("\nStarting parallel buffer computation...")
start_time = time.time()

results = Parallel(n_jobs=-1, backend="threading", verbose=10)(
    delayed(process_reach)(row["reach_id"], row.geometry, row["width"])
    for _, row in gdf.iterrows()
)

elapsed = time.time() - start_time

# Unpack results and populate GeoDataFrame
reach_ids = [r[0] for r in results]
buffer_geoms = [r[1] for r in results]

gdf_out = gdf.copy()
gdf_out.pop("geometry")
gdf_out["buffer_geometry"] = buffer_geoms
gdf_out = gdf_out.set_geometry("buffer_geometry")
gdf_out.crs = crs_meters

# Count successful/failed
success_count = sum(1 for b in buffer_geoms if b is not None)
failed_count = len(buffer_geoms) - success_count

print(f"\n{'='*70}")
print(f"Processing complete!")
print(f"  Total time: {elapsed:.1f} seconds (~{elapsed/60:.1f} minutes)")
print(f"  Successful: {success_count:,} reaches")
print(f"  Failed: {failed_count:,} reaches")
print(f"  Average: {elapsed/len(gdf)*1000:.2f} ms/reach")
print(f"{'='*70}\n")

# Save to file (meters CRS)
output_path_meters = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_corridor_buffer_all.gpkg"
# output_path_meters = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_width.gpkg"
gdf_out.to_file(output_path_meters, driver="GPKG")
print(f"Saved (meters): {output_path_meters}")

# Save to file (WGS84)
output_path_wgs84 = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_corridor_buffer_wgs84_all.gpkg"
# output_path_wgs84 = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_width_wgs84.gpkg"
gdf_out.to_crs("EPSG:4326").to_file(output_path_wgs84, driver="GPKG")
print(f"Saved (WGS84): {output_path_wgs84}")

print(f"\n All buffers successfully saved!")

Processing 38478 river reaches with parallel buffering

Starting parallel buffer computation...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    5.2s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:   13.0s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:   23.1s
[Parallel(n_jobs=-1)]: Done  66 tasks      | elapsed:   28.0s
[Parallel(n_jobs=-1)]: Done  85 tasks      | elapsed:   41.2s
[Parallel(n_jobs=-1)]: Done 104 tasks      | elapsed:   48.1s
[Parallel(n_jobs=-1)]: Done 125 tasks      | elapsed:   52.2s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 169 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 217 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 242 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 269 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 296 tasks      | elaps

## NLCD zonal extraction

### Load buffered polygons and NLCD

In [ ]:
"""
NLCD Zonal Extraction Script
- LndCov: Calculate percentage of 21, 22, 23, 24 land cover categories
- FctImp: Calculate mean impervious surface fraction
"""

# ============================================================================
# 1. File path configuration
# ============================================================================

# Input: existing buffered GeoPackage file
input_gdf_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_corridor_buffer_wgs84_all.gpkg"

# NLCD data files
nlcd_landcov_path = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1.tif"
nlcd_fctimp_path = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1.tif"

# Output: result save path
output_gdf_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_reaches_corridor_buffer_wgs84_all_NLCD.gpkg"

In [ ]:
# ============================================================================
# 2. Load GeoDataFrame and fix invalid geometries
# ============================================================================
print("Loading buffered GeoDataFrame...")
gdf = gpd.read_file(input_gdf_path)
print(f"Loaded {len(gdf)} geometries")

# Fix only invalid geometries
invalid_mask = ~gdf.geometry.is_valid
n_invalid = invalid_mask.sum()

if n_invalid > 0:
    print(f"Found {n_invalid} invalid geometries ({n_invalid/len(gdf)*100:.2f}%). Fixing...")
    gdf.loc[invalid_mask, 'geometry'] = gdf.loc[invalid_mask, 'geometry'].buffer(0)
    print(f"Fixed! Remaining invalid: {(~gdf.geometry.is_valid).sum()}")
else:
    print("All geometries are valid!")

print(f"Columns: {list(gdf.columns)}")
gdf['reach_id_str'] = gdf['reach_id'].astype(str)

###  Check NLCD CRS and align projections

In [ ]:
# ============================================================================
# 3. Check NLCD CRS and align projections
# ============================================================================

print("\n" + "="*70)
print("Checking and aligning CRS with NLCD rasters...")
print("="*70)

# Check LndCov CRS
with rasterio.open(nlcd_landcov_path) as src:
    nlcd_crs = src.crs
    print(f"NLCD LndCov CRS: {nlcd_crs}")

# Check FctImp CRS
with rasterio.open(nlcd_fctimp_path) as src:
    nlcd_fctimp_crs = src.crs
    print(f"NLCD FctImp CRS: {nlcd_fctimp_crs}")

# Reproject GeoDataFrame to match NLCD CRS if necessary
if gdf.crs != nlcd_crs:
    print(f"\n⚠ CRS mismatch detected!")
    print(f"  GeoDataFrame CRS: {gdf.crs}")
    print(f"  NLCD CRS: {nlcd_crs}")
    print(f"  Reprojecting GeoDataFrame to match NLCD CRS...")
    gdf = gdf.to_crs(nlcd_crs)
    print(f"  ✓ Reprojection completed")
    print(f"  Updated GeoDataFrame CRS: {gdf.crs}")
else:
    print(f"✓ CRS already matches NLCD CRS")

### Create binary masks for urban land cover categories (21-24)

In [12]:
# ============================================================================
# 4. Create binary masks for urban land cover categories (21-24)
# ============================================================================
# Category codes:
# 21 = Developed, Open Space
# 22 = Developed, Low Intensity
# 23 = Developed, Medium Intensity  
# 24 = Developed, High Intensity

output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_masks"
urban_categories = [21, 22, 23, 24]

# Read NLCD raster and prepare output profile
with rasterio.open(nlcd_landcov_path) as src:
    nlcd_data = src.read(1)
    output_profile = src.profile.copy()
    output_profile.update(
        compress='lzw',
        dtype='uint8',
        nodata=255
    )
    
    # Generate binary mask for each urban category
    for category in urban_categories:
        binary_mask = (nlcd_data == category).astype('uint8')
        output_path = f"{output_dir}/nlcd_{category}.tif"
        
        with rasterio.open(output_path, 'w', **output_profile) as dst:
            dst.write(binary_mask, 1)
        
        print(f"Created: nlcd_{category}.tif")

print("All binary masks created successfully!")

Created: nlcd_21.tif
Created: nlcd_22.tif
Created: nlcd_23.tif
Created: nlcd_24.tif
All binary masks created successfully!


### Process LndCov

In [9]:
# ============================================================================
# 5. Process LndCov: Calculate mean (batch processing)
# ============================================================================

nlcd_mask_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_masks"
output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_exactextract"
Path(output_dir).mkdir(parents=True, exist_ok=True)

urban_categories = [21, 22, 23, 24]
mask_paths = [f"{nlcd_mask_dir}/nlcd_{cat}.tif" for cat in urban_categories]

# Process in batches
batch_size = 100
n_total = len(gdf)
n_batches = (n_total + batch_size - 1) // batch_size

print(f"Processing {n_total} polygons in {n_batches} batches...")

for i in range(n_batches):
    clear_output(wait=True)
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_total)
    gdf_batch = gdf.iloc[start_idx:end_idx]
    
    stats = exact_extract(
        mask_paths,
        gdf_batch,
        ['sum', 'mean'],
        include_cols=['reach_id_str'],
        output='pandas'
    )
    
    output_path = f"{output_dir}/batch_{i:04d}.parquet"
    stats.to_parquet(output_path)
    
    print(f"Batch {i+1}/{n_batches} complete ({end_idx}/{n_total} polygons)")

print("All batches processed!")

Batch 385/385 complete (38478/38478 polygons)
All batches processed!


In [ ]:
output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_exactextract"
batch_files = sorted(Path(output_dir).glob("batch_*.parquet"))

# Read all at once
lndcov_stats_combined = pd.concat([pd.read_parquet(f) for f in batch_files], 
                           axis=0, ignore_index=True)

print(f"Total rows: {len(lndcov_stats_combined):,}")
lndcov_stats_combined.to_parquet(f"{output_dir}/nlcd_stats_combined.parquet")

#### NLCD exactextract troubleshooting

In [ ]:
# ============================================================================
# Debug batch 89 - find problematic polygon
# ============================================================================
import traceback

# ============================================================================
# 5. Process LndCov: Calculate mean (batch processing)
# ============================================================================

nlcd_mask_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_masks"
output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_exactextract"
Path(output_dir).mkdir(parents=True, exist_ok=True)

urban_categories = [21, 22, 23, 24]
mask_paths = [f"{nlcd_mask_dir}/nlcd_{cat}.tif" for cat in urban_categories]

# Process in batches
batch_size = 100
n_total = len(gdf)
n_batches = (n_total + batch_size - 1) // batch_size

print(f"Processing {n_total} polygons in {n_batches} batches...")

batch_idx = 89
start_idx = batch_idx * 100
end_idx = min((batch_idx + 1) * 100, n_total)

print(f"Testing batch {batch_idx} polygons ({start_idx} to {end_idx})...")

# Test each polygon individually
problematic_indices = []

for idx in range(start_idx, end_idx):
    clear_output(wait=True)
    try:
        gdf_single = gdf.iloc[[idx]]
        
        stats = exact_extract(
            mask_paths,
            gdf_single,
            ['sum', 'mean'],
            include_cols=['reach_id_str'],
            output='pandas'
        )
        
        print(f"✓ Polygon {idx} (reach_id: {gdf.iloc[idx]['reach_id_str']}) OK")
        
    except Exception as e:
        print(f"✗ Polygon {idx} FAILED!")
        print(f"  reach_id: {gdf.iloc[idx]['reach_id_str']}")
        print(f"  Error: {str(e)}")
        problematic_indices.append(idx)

print(f"\n{'='*80}")
print(f"Found {len(problematic_indices)} problematic polygons: {problematic_indices}")

# Examine problematic polygons
if problematic_indices:
    print(f"\n{'='*80}")
    print("Problematic polygon details:")
    for idx in problematic_indices:
        row = gdf.iloc[idx]
        print(f"\nIndex {idx}:")
        print(f"  reach_id: {row['reach_id_str']}")
        print(f"  geometry type: {row.geometry.geom_type}")
        print(f"  is valid: {row.geometry.is_valid}")
        print(f"  area: {row.geometry.area}")
        print(f"  bounds: {row.geometry.bounds}")

Processing 38478 polygons in 385 batches...
Testing batch 89 polygons (8900 to 9000)...
✓ Polygon 8900 (reach_id: 72555600071) OK
✓ Polygon 8901 (reach_id: 72555600081) OK
✓ Polygon 8902 (reach_id: 72555600091) OK
✓ Polygon 8903 (reach_id: 72555600101) OK
✓ Polygon 8904 (reach_id: 72555600111) OK
✓ Polygon 8905 (reach_id: 72555600124) OK
✓ Polygon 8906 (reach_id: 72555600131) OK
✓ Polygon 8907 (reach_id: 72555600144) OK
✓ Polygon 8908 (reach_id: 72555600151) OK
✓ Polygon 8909 (reach_id: 72555600161) OK
✓ Polygon 8910 (reach_id: 72555600174) OK
✓ Polygon 8911 (reach_id: 72555600181) OK
✓ Polygon 8912 (reach_id: 72555600194) OK
✓ Polygon 8913 (reach_id: 72555600201) OK
✓ Polygon 8914 (reach_id: 72555600211) OK
✓ Polygon 8915 (reach_id: 72555600221) OK
✓ Polygon 8916 (reach_id: 72555600341) OK
✓ Polygon 8917 (reach_id: 72555600351) OK
✓ Polygon 8918 (reach_id: 72555600231) OK
✓ Polygon 8919 (reach_id: 72555600241) OK
✓ Polygon 8920 (reach_id: 72555600251) OK
✓ Polygon 8921 (reach_id: 7255

In [9]:
# Check polygon 8934
idx = 8934
row = gdf.iloc[idx]

print(f"Problematic polygon at index {idx}:")
print(f"  reach_id: {row['reach_id']}")
print(f"  reach_id_str: {row['reach_id_str']}")
print(f"  geometry type: {row.geometry.geom_type}")
print(f"  is valid: {row.geometry.is_valid}")
print(f"  area: {row.geometry.area}")
print(f"  bounds: {row.geometry.bounds}")

Problematic polygon at index 8934:
  reach_id: 72555700123
  reach_id_str: 72555700123
  geometry type: MultiPolygon
  is valid: False
  area: 15.040620677451765
  bounds: (-82.57229151962918, 39.804362536524145, -81.62962685018681, 43.86100701059248)


In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import GeometryCollection

# Original and fixed geometries
idx = 8934
original = gdf.iloc[idx].geometry
fixed = original.buffer(0)

print(f"Original is valid: {original.is_valid}")
print(f"Fixed is valid: {fixed.is_valid}")
print(f"Original area: {original.area:.2f}")
print(f"Fixed area: {fixed.area:.2f}")
print(f"Area difference: {abs(original.area - fixed.area):.6f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot original
ax1 = axes[0]
if original.geom_type == 'MultiPolygon':
    for poly in original.geoms:
        x, y = poly.exterior.xy
        ax1.plot(x, y, 'r-', linewidth=2)
        ax1.fill(x, y, alpha=0.3, fc='red')
else:
    x, y = original.exterior.xy
    ax1.plot(x, y, 'r-', linewidth=2)
    ax1.fill(x, y, alpha=0.3, fc='red')
ax1.set_title(f'Original (reach_id: {gdf.iloc[idx]["reach_id_str"]})\nValid: {original.is_valid}', fontsize=14)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Plot fixed
ax2 = axes[1]
if fixed.geom_type == 'MultiPolygon':
    for poly in fixed.geoms:
        x, y = poly.exterior.xy
        ax2.plot(x, y, 'b-', linewidth=2)
        ax2.fill(x, y, alpha=0.3, fc='blue')
else:
    x, y = fixed.exterior.xy
    ax2.plot(x, y, 'b-', linewidth=2)
    ax2.fill(x, y, alpha=0.3, fc='blue')
ax2.set_title(f'Fixed with buffer(0)\nValid: {fixed.is_valid}', fontsize=14)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSaved comparison to: /work/pi_kandread_umass_edu/swot-urban/1_data_processing/geometry_fix_comparison.png")

### Process FctImp

In [10]:
# ============================================================================
# 6. Process FctImp: Calculate mean (batch processing)
# ============================================================================

output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_exactextract_FctImp"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Process in batches
batch_size = 100
n_total = len(gdf)
n_batches = (n_total + batch_size - 1) // batch_size

print(f"Processing {n_total} polygons in {n_batches} batches...")

for i in range(n_batches):
    clear_output(wait=True)
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_total)
    gdf_batch = gdf.iloc[start_idx:end_idx]
    
    stats = exact_extract(
        nlcd_fctimp_path,
        gdf_batch,
        ['count', 'mean'],
        include_cols=['reach_id_str'],
        output='pandas'
    )
    
    output_path = f"{output_dir}/FctImp_batch_{i:04d}.parquet"
    stats.to_parquet(output_path)
    
    print(f"Batch {i+1}/{n_batches} complete ({end_idx}/{n_total} polygons)")

print("All batches processed!")

Batch 385/385 complete (38478/38478 polygons)
All batches processed!


In [ ]:
output_dir = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/nlcd_exactextract_FctImp"
batch_files = sorted(Path(output_dir).glob("FctImp_batch_*.parquet"))

# Read all at once
fctimp_stats_combined = pd.concat([pd.read_parquet(f) for f in batch_files], 
                           axis=0, ignore_index=True)

print(f"Total rows: {len(fctimp_stats_combined):,}")
fctimp_stats_combined.to_parquet(f"{output_dir}/FctImp_nlcd_stats_combined.parquet")

###  Merge urban statistics with buffered polygons

In [ ]:
# ============================================================================
# 7. Merge urban statistics with buffered polygons
# ============================================================================

# Rename FctImp columns before merging
fctimp_stats_combined = fctimp_stats_combined.rename(columns={
    'count': 'fctimp_count',
    'mean': 'fctimp_mean'
})

# Merge buffered polygons with urban categories and fractional impervious
gdf_merged = gdf.merge(lndcov_stats_combined, on='reach_id_str', how='left')
gdf_merged = gdf_merged.merge(fctimp_stats_combined, on='reach_id_str', how='left')

print(f"Merged: {len(gdf_merged):,} rows")
print(f"Columns: {gdf_merged.columns.tolist()}")

print(f"\n Final merged GeoDataFrame:")
print(f"  Total rows: {len(gdf_merged):,}")
print(f"  Geometry type: {gdf_merged.geometry.geom_type.unique()}")
print(f"\nNew columns added:")
new_cols = [col for col in gdf_merged.columns if col not in gdf.columns]
print(new_cols)

# Save
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/buffered_polygons_with_nlcd.shp"
gdf_merged.to_file(output_path)
print(f"\n Saved to: {output_path}")